# Project Lab 2 — Mock-Interview Kit and Resume Sprint

Notebook for **Chapter 16** of the *career* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/career/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Mock-Interview Kit and Answer Bank

In [ ]:
"""Mock-interview kit. Commands: init | add | quiz | log | report"""
import argparse, random, sqlite3, sys
from datetime import date

DB = "kit.db"
SCHEMA = """
CREATE TABLE IF NOT EXISTS questions(
  id INTEGER PRIMARY KEY, topic TEXT NOT NULL, question TEXT NOT NULL,
  model_answer TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS attempts(
  id INTEGER PRIMARY KEY, question_id INTEGER NOT NULL REFERENCES questions(id),
  day TEXT NOT NULL, clarity INTEGER, structure INTEGER, evidence INTEGER,
  CHECK (clarity BETWEEN 1 AND 5 AND structure BETWEEN 1 AND 5 AND evidence BETWEEN 1 AND 5));
"""
SEED = [
 ("behavioral", "Tell me about a time you disagreed with a teammate.", "STAR: situation, task, action (listened, tested both ideas), result (with a number)."),
 ("behavioral", "Describe a mistake you made and what you changed.", "Own the mistake early, show the fix, show the habit that now prevents it."),
 ("sql", "What is the difference between WHERE and HAVING?", "WHERE filters rows before grouping; HAVING filters groups after aggregation."),
 ("sql", "How do you find duplicate order ids?", "GROUP BY order_id HAVING COUNT(*) > 1."),
 ("stats", "What does a p-value of 0.03 mean?", "If there were no real effect, we would see a result this extreme about 3 percent of the time."),
 ("project", "Walk me through your best project in two minutes.", "Problem, data, method, result with a number, one limit, one next step."),
]

def connect():
    con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
    con.executescript(SCHEMA); return con

def cmd_init(a):
    con = connect()
    if con.execute("SELECT COUNT(*) FROM questions").fetchone()[0] == 0:
        con.executemany("INSERT INTO questions(topic,question,model_answer) VALUES(?,?,?)", SEED)
        con.commit()
    print("questions:", con.execute("SELECT COUNT(*) FROM questions").fetchone()[0])

def cmd_add(a):
    con = connect()
    con.execute("INSERT INTO questions(topic,question,model_answer) VALUES(?,?,?)", (a.topic, a.question, a.answer))
    con.commit(); print("added")

def pick(con, n, rng):
    """Weakest-first: questions never tried or with a low average score are likelier."""
    rows = con.execute("""
      SELECT q.id, q.topic, q.question, q.model_answer,
             COALESCE(AVG((at.clarity+at.structure+at.evidence)/3.0), 0) AS avg_score
      FROM questions q LEFT JOIN attempts at ON at.question_id = q.id
      GROUP BY q.id""").fetchall()
    weights = [6 - r["avg_score"] if r["avg_score"] else 5 for r in rows]
    chosen = []
    pool = list(zip(rows, weights))
    for _ in range(min(n, len(pool))):
        r = rng.choices([p[0] for p in pool], [p[1] for p in pool])[0]
        chosen.append(r); pool = [p for p in pool if p[0]["id"] != r["id"]]
    return chosen

def cmd_quiz(a):
    con = connect(); rng = random.Random(a.seed)
    for r in pick(con, a.n, rng):
        print(f"\n[{r['topic']}] Q{r['id']}: {r['question']}")
        if a.demo:
            print("  (demo mode: answer aloud, then compare)")
            print("  Model answer:", r["model_answer"])
            continue
        input("  Answer aloud, then press Enter to see the model answer... ")
        print("  Model answer:", r["model_answer"])
        s = [int(input(f"  Score {k} 1-5: ")) for k in ("clarity", "structure", "evidence")]
        con.execute("INSERT INTO attempts(question_id,day,clarity,structure,evidence) VALUES(?,?,?,?,?)",
                    (r["id"], str(date.today()), *s)); con.commit()

def cmd_log(a):
    con = connect()
    con.execute("INSERT INTO attempts(question_id,day,clarity,structure,evidence) VALUES(?,?,?,?,?)",
                (a.qid, a.day, a.clarity, a.structure, a.evidence)); con.commit(); print("logged")

def cmd_report(a):
    con = connect()
    print(f"{'topic':<11}{'tries':>6}{'avg':>6}")
    for r in con.execute("""SELECT q.topic, COUNT(at.id) n,
        ROUND(AVG((at.clarity+at.structure+at.evidence)/3.0),2) avg
        FROM questions q LEFT JOIN attempts at ON at.question_id=q.id
        GROUP BY q.topic ORDER BY avg IS NULL, avg"""):
        print(f"{r['topic']:<11}{r['n']:>6}{(r['avg'] if r['avg'] is not None else '-'):>6}")
    ready = con.execute("""SELECT COUNT(*) FROM (SELECT question_id FROM attempts
        GROUP BY question_id HAVING AVG((clarity+structure+evidence)/3.0) >= 4)""").fetchone()[0]
    total = con.execute("SELECT COUNT(*) FROM questions").fetchone()[0]
    print(f"interview-ready questions (avg >= 4): {ready} of {total}")

p = argparse.ArgumentParser(); sub = p.add_subparsers(dest="cmd", required=True)
sub.add_parser("init").set_defaults(f=cmd_init)
s = sub.add_parser("add"); [s.add_argument(x) for x in ("topic", "question", "answer")]; s.set_defaults(f=cmd_add)
s = sub.add_parser("quiz"); s.add_argument("-n", type=int, default=3); s.add_argument("--seed", type=int)
s.add_argument("--demo", action="store_true"); s.set_defaults(f=cmd_quiz)
s = sub.add_parser("log"); s.add_argument("qid", type=int); s.add_argument("day")
[s.add_argument(x, type=int) for x in ("clarity", "structure", "evidence")]; s.set_defaults(f=cmd_log)
sub.add_parser("report").set_defaults(f=cmd_report)
a = p.parse_args(); a.f(a)

## 16.2 Project: Resume and LinkedIn Refresh Sprint

In [ ]:
"""Flag weak resume bullets. Usage: python3 check_bullets.py bullets.txt"""
import re, sys

WEAK_STARTS = ("responsible for", "worked on", "helped", "participated", "involved in",
               "assisted", "duties included", "tasked with")
FILLER = ("various", "etc", "multiple things", "many", "several", "hard-working")
STRONG_VERBS = {"built", "wrote", "cut", "reduced", "automated", "designed", "launched",
                "analysed", "analyzed", "migrated", "created", "improved", "led", "fixed", "tested"}

def check(b):
    low, issues = b.lower().strip(), []
    if low.startswith(WEAK_STARTS):
        issues.append("weak opening phrase")
    elif low.split()[0] not in STRONG_VERBS:
        issues.append("does not start with a strong verb")
    if not re.search(r"\d", b):
        issues.append("no number")
    if any(re.search(rf"\b{re.escape(w)}\b", low) for w in FILLER):
        issues.append("filler word")
    n = len(b.split())
    if n < 8: issues.append(f"too short ({n} words)")
    if n > 30: issues.append(f"too long ({n} words)")
    return issues

bullets = [l.strip() for l in open(sys.argv[1]) if l.strip()]
flagged = 0
for i, b in enumerate(bullets, 1):
    issues = check(b)
    flagged += bool(issues)
    print(f"{i}. [{'FIX' if issues else 'OK '}] {b[:60]}{'...' if len(b) > 60 else ''}")
    for x in issues: print(f"      - {x}")
print(f"\n{len(bullets) - flagged} of {len(bullets)} bullets pass")
sys.exit(1 if flagged else 0)

## 16.3 Scored Sample Resumes: Before and After

## Chapter recap

## Practice set

**1.** (Easy) Why does the `attempts` table store `question_id` instead of the question text? **Answer:** to avoid repeating the text and to link many attempts to one question; a join brings the text back when needed.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/career/ch16.html